# 05 — Single final test session (§0.7 — run ONCE)

**The test set is evaluated once, at the end, with the val-selected checkpoint
of each of the 16 pre-registered rows** (frozen §0.7 list, amended
2026-07-19/20/21; `C1_aug_s43` dropped 2026-07-22, stress-test L6). Every access
goes through the logging harness, so each execution appends to
`test_invocations.jsonl` — re-runs are visible forever.

**Two modes, `SET` below:**
- `SET = "val"` (**default — a DRY RUN**): exercises the entire 16-row path on
  the val sets. The harness logs a test invocation ONLY on `set_name=="test"`
  (verified: `harness.py` lines 354/440/512), so a val run writes **no**
  `test_invocations.jsonl` entry and is free to re-run. If the dry run completes
  clean and the per-row summary is sane, the test session will too. This is the
  de-risk step — run it first.
- `SET = "test"`: the real, once-only session. Requires flipping `SET` **and**
  setting `I_CONFIRM_SINGLE_TEST_SESSION = True`.

Nothing here selects a checkpoint or changes a split — selection happened on
val, upstream. This notebook only reads frozen artifacts and evaluates.

## Environment setup (repo + Drive + staging)

Staging is needed: `evaluate`/`cache_features` run the encoder over real data. All outputs (CSVs, feature caches, the test-invocation log) go to a local `SESSION_DIR`, so the run needs only **read** access to the Drive checkpoints — it never writes to the run folders.

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

import numpy as np
import torch

REPO_URL = "https://github.com/FilippoIsoni/sharp-har.git"

cwd = Path.cwd().resolve()
if (cwd / "sharp_har").exists():
    REPO_DIR = cwd
elif (cwd.parent / "sharp_har").exists():
    REPO_DIR = cwd.parent
else:
    REPO_DIR = Path("/content/sharp-har")
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

get_ipython().system("pip install -q -r {}/requirements.txt".format(REPO_DIR))

sys.path.insert(0, str(REPO_DIR))

from google.colab import drive
from sharp_har.utils import read_yaml

paths_cfg = read_yaml(REPO_DIR / "configs" / "paths.yaml")
drive_root = Path(paths_cfg["drive_root"])
stage_dir = Path(paths_cfg["stage_dir"])
CKPT_ROOT = Path(paths_cfg["ckpt_root"])

drive.mount("/content/drive")

if not (stage_dir.exists() and any(stage_dir.rglob("*.txt"))):
    stage_dir.mkdir(parents=True, exist_ok=True)
    for zip_name in paths_cfg["zips"]:
        src = drive_root / zip_name
        dst = Path("/content") / zip_name
        print("copying", src, "->", dst)
        subprocess.run(["cp", str(src), str(dst)], check=True)
        with zipfile.ZipFile(dst) as zf:
            zf.extractall(stage_dir)

print("Repo:", REPO_DIR, "| GPU:", torch.cuda.is_available(), "| CKPT_ROOT:", CKPT_ROOT)

## Mode selector — DEFAULT IS THE VAL DRY RUN

Flip to `"test"` only for the real, once-only session, and only after a clean dry run.

In [ ]:
SET = "val"   # "val" = dry run (no test contact) | "test" = the single §0.7 session
I_CONFIRM_SINGLE_TEST_SESSION = False   # must be True to run SET="test"

assert SET in ("val", "test")
if SET == "test":
    assert I_CONFIRM_SINGLE_TEST_SESSION, (
        "SET='test' is the ONE-TIME §0.7 session. Set I_CONFIRM_SINGLE_TEST_SESSION=True "
        "to proceed — after a clean val dry run. Every row will append to test_invocations.jsonl."
    )
    print("*** REAL TEST SESSION — this evaluates the held-out test sets ONCE ***")
else:
    print("DRY RUN on val — no test contact, no test_invocations.jsonl entry, safe to re-run.")

# Per-row output staging (kept off the Drive run folders so a re-run is clean).
SESSION_DIR = Path("/content/session_out") / SET
SESSION_DIR.mkdir(parents=True, exist_ok=True)
P2_LAB = REPO_DIR / "splits" / "p2_lab.json"
P2_LIVING = REPO_DIR / "splits" / "p2_living.json"
P1_SHARP = REPO_DIR / "splits" / "p1_sharp.json"

## The frozen 16-row list + readiness assert

The rows are declared once, here — the §0.7 hard-coded list. The assert checks
every required artifact (checkpoint / probe head / phase-B selection) exists on
Drive **before any evaluation runs**; a missing one stops the session.

Row kinds:
- `e2e` — end-to-end CE checkpoint via `evaluate` (optional `adapt_bn`).
- `c0` — `evaluate_c0` (paper decision fusion, P1, 5-class).
- `probe` — cache features then `evaluate_features` with a persisted linear head.
- `t3a` — cache features (raw or post-AdaBN) then a T3A prototype head.

The aug arm's seed twin `C1_aug_s43` was dropped (stress-test L6, ratified
2026-07-22, CHANGELOG): the paired design already controls the init, and the s43
twin re-uses the S7 test set, so it adds no independent replication over `C1_aug`
(the cross-rotation `C1_s6out_aug` twin is the one kept). The list is 16 rows;
the assert adapts to whatever is declared here.

In [ ]:
# key -> row spec. folder = Drive run folder under CKPT_ROOT.
ROWS = [
    dict(key="C0",            kind="c0",    folder="C0",            split=P1_SHARP),
    dict(key="C1",            kind="e2e",   folder="C1",            split=P2_LAB),
    dict(key="C1_s43",        kind="e2e",   folder="C1_s43",        split=P2_LAB),
    dict(key="C2",            kind="e2e",   folder="C2",            split=P2_LAB),
    dict(key="C2_s43",        kind="e2e",   folder="C2_s43",        split=P2_LAB),
    dict(key="C1_lin",        kind="probe", folder="C1", ckpt="best.ckpt",   head="probe_head_best.npz",    split=P2_LAB),
    dict(key="C2_lin",        kind="probe", folder="C2", ckpt="best.ckpt",   head="probe_head_best.npz",    split=P2_LAB),
    dict(key="C3",            kind="probe", folder="C3", from_phaseb=True,                                  split=P2_LAB),
    dict(key="C3_ft",         kind="e2e",   folder="C3_ft",         split=P2_LAB),
    dict(key="C1_AdaBN",      kind="e2e",   folder="C1", adapt_bn=True,       split=P2_LAB),
    dict(key="C1_T3A",        kind="t3a",   folder="C1", adapt_bn=False,      split=P2_LAB),
    dict(key="C1_AdaBN_T3A",  kind="t3a",   folder="C1", adapt_bn=True,       split=P2_LAB),
    dict(key="C1_s6out",      kind="e2e",   folder="C1_s6out",      split=P2_LIVING),
    dict(key="C1_aug",        kind="e2e",   folder="C1_aug",        split=P2_LAB),
    dict(key="C1_s6out_aug",  kind="e2e",   folder="C1_s6out_aug",  split=P2_LIVING),
    dict(key="C1_sharplike",  kind="e2e",   folder="C1_sharplike",  split=P2_LAB),
]
print(f"{len(ROWS)} rows declared (frozen §0.7 list)")

import json
def required_paths(r):
    d = CKPT_ROOT / r["folder"]
    if r["kind"] == "probe" and r.get("from_phaseb"):
        sel = d / "phase_b_selection.json"
        if not sel.exists():
            return [sel]
        # Rebuild from CKPT_ROOT + selected_epoch — the stored selected_checkpoint/
        # selected_head are ABSOLUTE paths from the session that made the file and
        # may not resolve under a different mount/shortcut. Filenames are deterministic.
        e = json.loads(sel.read_text())["selected_epoch"]
        return [d / f"epoch{e}.ckpt", d / f"probe_head_epoch{e}.npz"]
    if r["kind"] == "probe":
        return [d / r["ckpt"], d / r["head"]]
    return [d / "best.ckpt"]   # e2e, c0, t3a all need best.ckpt

missing = []
for r in ROWS:
    for p in required_paths(r):
        if not Path(p).exists():
            missing.append((r["key"], str(p)))
if missing:
    print("READINESS FAILED — missing artifacts:")
    for k, p in missing:
        print(f"  [{k}] {p}")
    raise SystemExit("Not ready: every row needs its val-selected artifact on Drive "
                     "(add Editor shortcuts to EVERY run folder from this one account).")
print("READINESS OK — all", len(ROWS), "rows have their artifacts.")

## Run the 16 rows

Each row writes windows/metrics/confusion CSVs into its own `SESSION_DIR/<key>/`; the summary reads the fused accuracy back for a sanity pass (this is what makes the dry run self-verifying).

In [ ]:
from sharp_har.harness import evaluate, evaluate_c0, evaluate_features, cache_features
from sharp_har.transductive import t3a_head, head_weight_from_checkpoint
import json
import pandas as pd

def _finalize(key, out_dir, fusion):
    """Copy the harness CSVs to the report dir with the naming contract
    notebook 06 expects: <key>_<SET>_<fusion>_<kind>.csv. The real session
    writes to reports/final/ (committed); the val dry run writes to a
    throwaway preview dir under SESSION_DIR so the repo tree stays clean
    (the naming/copy path is still exercised, just not into the commit dir)."""
    final = (REPO_DIR / "reports" / "final") if SET == "test" else (SESSION_DIR / "_final_preview")
    final.mkdir(parents=True, exist_ok=True)
    for csv in Path(out_dir).glob("*.csv"):
        kind = csv.stem.rsplit("_", 1)[1]  # windows | metrics | confusion
        (final / f"{key}_{SET}_{fusion}_{kind}.csv").write_bytes(csv.read_bytes())

def _acc(out_dir):
    w = next(Path(out_dir).glob("*_windows.csv"))
    df = pd.read_csv(w)
    return float((df.y_pred == df.y_true).mean()), df.trace_id.nunique(), len(df)

def run_row(r):
    d = CKPT_ROOT / r["folder"]
    out = SESSION_DIR / r["key"]; out.mkdir(parents=True, exist_ok=True)
    fusion = "sharp_c0" if r["kind"] == "c0" else "softmax_avg"

    if r["kind"] == "c0":
        evaluate_c0(d / "best.ckpt", r["split"], SET, stage_dir=stage_dir, out_dir=out, repo_dir=REPO_DIR)

    elif r["kind"] == "e2e":
        evaluate(d / "best.ckpt", r["split"], SET, stage_dir=stage_dir, out_dir=out,
                 repo_dir=REPO_DIR, adapt_bn=r.get("adapt_bn", False))

    elif r["kind"] == "probe":
        if r.get("from_phaseb"):
            e = json.loads((d / "phase_b_selection.json").read_text())["selected_epoch"]
            ckpt, head_path = d / f"epoch{e}.ckpt", d / f"probe_head_epoch{e}.npz"
        else:
            ckpt, head_path = d / r["ckpt"], d / r["head"]
        feat = cache_features(ckpt, r["split"], SET, stage_dir=stage_dir, repo_dir=REPO_DIR,
                              out_path=out / f"features_{SET}.npz")
        hz = np.load(head_path, allow_pickle=False)
        head = {"weight": hz["weight"], "bias": hz["bias"]}
        evaluate_features(feat, head, SET, out_dir=out, run_name=r["key"], repo_dir=REPO_DIR)

    elif r["kind"] == "t3a":
        ckpt = d / "best.ckpt"
        feat = cache_features(ckpt, r["split"], SET, stage_dir=stage_dir, repo_dir=REPO_DIR,
                              adapt_bn=r.get("adapt_bn", False), out_path=out / f"features_{SET}.npz")
        raw = np.load(feat, allow_pickle=False)["features"]
        head = t3a_head(head_weight_from_checkpoint(ckpt), raw)  # prototypes from C1's head; features raw/post-AdaBN
        head = {"weight": head["weight"], "bias": head["bias"]}
        evaluate_features(feat, head, SET, out_dir=out, run_name=r["key"], repo_dir=REPO_DIR)

    else:
        raise ValueError(r["kind"])

    _finalize(r["key"], out, fusion)
    return _acc(out)

summary = []
for r in ROWS:
    acc, ntr, nw = run_row(r)
    summary.append((r["key"], acc, ntr, nw))
    print(f"  {r['key']:16s} acc {acc:.4f}  ({ntr} traces, {nw} fused windows)")
print("\nall rows done on", SET)

## Summary table

Sanity pass. On the dry run: numbers near the val macro-F1s already on record (accuracy, not macro-F1, so not identical) and no crash = the path is sound. On the real session: this is the headline table (read alongside notebook 06's paired bootstrap + per-class decomposition).

In [ ]:
import pandas as pd
tab = pd.DataFrame(summary, columns=["row", "accuracy", "n_traces", "n_windows"])
print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
dest = "reports/final/" if SET == "test" else str(SESSION_DIR / "_final_preview")
print(f"\nSET = {SET} | CSVs in {dest} named <row>_{SET}_<fusion>_<kind>.csv (notebook 06 contract)")

## Commit the measured artifacts (real session only)

On `SET="test"`, commit **in one commit**: `reports/final/` (the per-row CSVs +
each run folder's `test_invocations.jsonl`), this executed notebook under
`notebooks/runs/`, and the `STATUS.md` update. Then run notebook 06 on
`reports/final/` for the paired bootstrap, calibration and per-class /
class-coverage decomposition (§9 report material).

In [ ]:
if SET == "test":
    final = REPO_DIR / "reports" / "final"
    logs = list(SESSION_DIR.glob("*/test_invocations.jsonl"))
    with open(final / "test_invocations.jsonl", "w", encoding="utf-8") as agg:
        for lg in sorted(logs):
            agg.write(lg.read_text(encoding="utf-8"))
    print(f"merged {len(logs)} per-row test-invocation logs into", final / "test_invocations.jsonl")
    print("now: git add reports/final + this notebook (notebooks/runs/) + STATUS, one commit.")
else:
    print("dry run — nothing to commit. Flip SET to 'test' (and confirm) for the real session.")